# Floor ranker with the rewrite, hosted twin (LLaMA 3.1-8B, DeepInfra fp8)

The rewrite condition under candidate injection with both rankers: 2 rankers x 3 passes x 2 query halves = **12 detached lanes**,
launched together so provider drift cannot separate the two rankers. Run cells 1 to 3, then close the tab if you like.
Watch progress in `floor_monitor.ipynb`. Cost: under $1.

In [1]:
# 1. Setup

import os, sys, subprocess, time, json, csv, io, zipfile, shutil, glob, urllib.request
csv.field_size_limit(10**8)

def _find_repo():
    # The unpacked harness: ~/work/skyrag, or the one folder under ~/work holding it (e.g. a project folder).
    base = os.path.expanduser("~/work")
    hits = [os.path.dirname(p) for p in glob.glob(os.path.join(base, "skyrag", "BUNDLE.json"))
            + glob.glob(os.path.join(base, "*", "skyrag", "BUNDLE.json"))]
    if len(hits) != 1:
        raise SystemExit(f"expected exactly one unpacked harness under {base}, found {hits or 'none'}")
    return hits[0]
REPO = _find_repo(); WORK = os.path.dirname(REPO); OUT = os.path.join(REPO, "outputs")
print("harness:", REPO)

os.makedirs(OUT, exist_ok=True)
print("bundle commit:", json.load(open(os.path.join(REPO, "BUNDLE.json")))["commit"][:7])
VLLM = shutil.which("vllm") or "/opt/conda/bin/vllm"
PY = os.path.join(os.path.dirname(VLLM), "python") if os.path.exists(VLLM) else sys.executable

# Two query halves of the benchmark, written once (the same files the earlier runs used).
src = [l for l in open(os.path.join(REPO, "benchmarks/queries_v1.jsonl"), encoding="utf-8").read().splitlines() if l.strip()]
HALVES = {1: "benchmarks/queries_v1_h1.jsonl", 2: "benchmarks/queries_v1_h2.jsonl"}
for k, part in ((1, src[:25]), (2, src[25:])):
    p = os.path.join(REPO, HALVES[k])
    if not os.path.exists(p):
        open(p, "w", encoding="utf-8").write("\n".join(part) + "\n")
RANKERS = ["lexical", "floor"]
ROWS = 25
def eval_args(qfile, precision, served, mode, ranker):
    return ["-Q", qfile, "-b", "openrouter", "-P", "skyrag", "--serving-precision", precision, "--sweep-version", "v5",
            "-m", served, "--arm", "2", "--candidate-injection", "--candidate-limit", "100", "--candidate-ranker", ranker,
            "-r", "1", *(["--expansion-mode", mode] if mode else [])]
def rows_of(name):
    try:
        with io.open(os.path.join(OUT, name, "benchmark_eval.csv"), encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    except Exception:
        return []

subprocess.run([PY, "-m", "pip", "install", "-q", "httpx>=0.27", "requests", "python-dotenv",
                "sentence-transformers>=2.5", "pandas>=2.2", "scipy", "tqdm"], check=True)
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)

harness: /home/jovyan/work/TA-Sean-Matthew/skyrag
bundle commit: b3153b8
               total        used        free      shared  buff/cache   available
Mem:             125           4          18           0         104         121
Swap:              7           0           7



In [2]:
# 2. OpenRouter key (memory only, never written to a file)
from getpass import getpass
if not os.environ.get("OPENROUTER_API_KEY") or os.environ["OPENROUTER_API_KEY"] == "local-vllm":
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ").strip()
print("key set:", bool(os.environ["OPENROUTER_API_KEY"]))

OpenRouter API key:  ········


key set: True


In [3]:
# 3. Launch all 12 lanes, detached. Safe to re-run: complete lanes are skipped.
ENV = {**os.environ, "SKYRAG_OPENROUTER_PROVIDER": "deepinfra", "PYTHONPATH": os.path.join(REPO, "src"),
       "SKYRAG_DEVICE": "cpu", "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
       "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("OPENROUTER_BASE_URL", None)    # must reach OpenRouter, not the local vLLM server
if subprocess.run(["pgrep", "-f", "skyrag.cli eval.*floorhosted"], capture_output=True, text=True).stdout.strip():
    raise SystemExit("Hosted lanes from an earlier start are still running. Wait for them, then run this cell again.")
n = 0
for rep in (1, 2, 3):
    for h, qfile in HALVES.items():
        for rk in RANKERS:                      # the two rankers of a pass start back to back
            name = f"floorhosted_20261007_v5fp8_i_a2_s1_{rk}_r{rep}_h{h}"
            out = os.path.join(OUT, name)
            if len(rows_of(name)) >= ROWS:
                print("skip (complete):", name); continue
            shutil.rmtree(out, ignore_errors=True)
            log = open(os.path.join(OUT, f"lane_{name}.log"), "w")
            subprocess.Popen([PY, "-m", "skyrag.cli", "eval", *eval_args(qfile, "fp8", "meta-llama/llama-3.1-8b-instruct", "schema", rk),
                              "-o", out + "/"], cwd=REPO, env=ENV, stdout=log, stderr=subprocess.STDOUT,
                             stdin=subprocess.DEVNULL, start_new_session=True)
            n += 1; print(time.strftime("%H:%M:%S"), "launched", name); time.sleep(8)
print(n, "lanes launched. Close the tab if you like; watch floor_monitor.ipynb.")

05:16:17 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r1_h1
05:16:25 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r1_h1
05:16:33 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r1_h2
05:16:41 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r1_h2
05:16:49 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r2_h1
05:16:57 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r2_h1
05:17:05 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r2_h2
05:17:13 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r2_h2
05:17:21 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r3_h1
05:17:29 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r3_h1
05:17:37 launched floorhosted_20261007_v5fp8_i_a2_s1_lexical_r3_h2
05:17:45 launched floorhosted_20261007_v5fp8_i_a2_s1_floor_r3_h2
12 lanes launched. Close the tab if you like; watch floor_monitor.ipynb.
